# 01 — Load the Lead Data

This notebook loads the insurance lead dataset and gives a first look at what's inside. It's the entry point for everything that follows.

## A note on the dataset

The CSV in `datasets/insurance_leadgen_data.csv` is a **prepared** version of an internal lead-generation dataset. The original was tidied up before being committed to this repo so it could be shared safely:

1. **Anonymized lead IDs** — the original IDs were replaced with random 8-digit numbers so individual records can't be traced.
2. **~5% of unverified leads removed** — the weakest part of the funnel was thinned out, mostly to shrink the file and remove low-signal rows.
3. **~10% of synthetic rows added** — extra rows were generated to balance the sample and round it out.
4. **Column names simplified** — converted to `snake_case` so they're easy to use in code.

All of this happened **before** this notebook runs. Treat the CSV as the source of truth for the project — there is no upstream raw file to fetch.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
DATASET_PATH = PROJECT_ROOT / "datasets" / "insurance_leadgen_data.csv"

leads = pd.read_csv(DATASET_PATH)
print(f"Loaded {leads.shape[0]:,} rows × {leads.shape[1]} columns from {DATASET_PATH.name}")

Loaded 7,485 rows × 13 columns from insurance_leadgen_data.csv


## 1 · First Look

In [2]:
leads.head()

,lead_id,lead_status,premium,age,gender,smoker,current_insurance,device_type,keyword,match_type,postcode,cover_for,verification_status
0,90809057,Contacted,0.0,31,Female,Yes,no,Desktop,private health insurance belfast,Exact,Bt13,Self,NaN
1,35622726,Contacted,0.0,35,Female,No,no,Smartphone,private medical insurance,Phrase,LU7,Self,NaN
2,29156003,Contacted,0.0,46,Male,No,yes private,Smartphone,private health insurance,Broad,E14,Self,NaN
3,26436791,No Contact,0.0,53,Female,No,no,Desktop,bupa health insurance,Exact,W2,Self + Partner,NaN
4,80270348,Contacted,0.0,35,Male,No,no,Smartphone,private healthcare prices,Exact,BH23,Self,NaN


In [3]:
print("Columns:")
for col in leads.columns:
    print(f"  - {col}  ({leads[col].dtype})")

Columns:
  - lead_id  (int64)
  - lead_status  (object)
  - premium  (float64)
  - age  (int64)
  - gender  (object)
  - smoker  (object)
  - current_insurance  (object)
  - device_type  (object)
  - keyword  (object)
  - match_type  (object)
  - postcode  (object)
  - cover_for  (object)
  - verification_status  (object)


## 2 · Lead Status Breakdown

`lead_status` is the headline outcome column — it tells us what happened to each lead in the funnel.

In [4]:
status_counts = leads["lead_status"].value_counts()
print(status_counts)
print()
print(f"Conversion rate (Sold ÷ total): {(status_counts.get('Sold', 0) / len(leads)):.2%}")

lead_status
Contacted     5968
Quoted         510
Sold           367
Invalid        326
No Contact     314
Name: count, dtype: int64

Conversion rate (Sold ÷ total): 4.90%


## 3 · Verification Status

Whether a lead was verified is one of the strongest signals in the dataset. Note that unverified leads come through as `NaN` here — we'll fix that in NB02.

In [5]:
print(leads["verification_status"].value_counts(dropna=False))

verification_status
NaN         5931
verified    1554
Name: count, dtype: int64


## 4 · Summary

We have a single CSV with ~7.5k rows describing insurance leads, their funnel status, the keyword that brought them in, and basic demographics. That's the only input we need.

**Next:** [02_clean_and_validate.ipynb](02_clean_and_validate.ipynb) standardises the columns, builds the target flags, and validates the result with Pandera before saving it as a parquet that every later notebook reads from.